# ARS Notebook 6 - Paper Figures and Result Archive

Create presentation-ready figures and tables from saved JSON/JSONL outputs. This notebook is CPU-only and does not load Phi-2 or repeat inference. Figures use a black background with white and orange marks to match the project deck.


## Input contract

Attach the completed result bundles from Notebooks 2-4. No model files or GPU are needed. The notebook records which input bundles were found and which conditions are missing; it never fills missing measurements with fabricated values.

In [ ]:
import json, os, subprocess, sys, zipfile, shutil
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
bundle_names = ("ars_gsm8k_results.zip", "ars_math_ood_results.zip", "ars_mcq_results.zip")
bootstrap_archives = [p for name in bundle_names for p in INPUT_ROOT.rglob(name)]
code_bootstrapped = False
for bootstrap_archive in bootstrap_archives:
    with zipfile.ZipFile(bootstrap_archive) as archive:
        if "code/scripts/research_notebook_utils.py" not in archive.namelist() or "code/scripts/research_notebook_analysis.py" not in archive.namelist():
            continue
        for name in ("research_notebook_utils.py", "research_notebook_analysis.py"):
            member = f"code/scripts/{name}"
            (REPO_DIR / "scripts" / name).write_bytes(archive.read(member))
        code_bootstrapped = True
        break
if not code_bootstrapped:
    raise FileNotFoundError("Attach a result bundle created by the new Notebook 2, 3, or 4; it carries the matching analysis code.")
from scripts.research_notebook_analysis import merge_available_results, build_router_diagnostics, create_paper_assets
RESULTS_ROOT = WORK_ROOT / "ars_all_saved_results"
INPUT_REPORT = merge_available_results([INPUT_ROOT], RESULTS_ROOT)
print(INPUT_REPORT)

## Generate assets

The analysis creates accuracy, paired-interval, wins/losses/ties, transfer-matrix, route-size, route-frequency, layer-frequency, alpha-frequency, diversity/concentration, completion-length, and per-question benefit/harm figures when the corresponding source data exists.

In [ ]:
ASSET_DIR = WORK_ROOT / "ars_paper_assets"
assets = create_paper_assets([RESULTS_ROOT], ASSET_DIR)
contact_sheet = ASSET_DIR / "figures" / "00_figure_contact_sheet.png"
if contact_sheet.is_file():
    from IPython.display import Image, display
    display(Image(filename=str(contact_sheet)))
print({"figures": assets["figures"], "figure_count": assets["figure_count"],
       "summary": assets["summary"], "bundle": assets["bundle"]})

## Export

Download `ars_paper_assets.zip` from the output directory. It contains the PNG figures, CSV tables, JSON summary, and a manifest with source-record paths and figure names. Save this bundle with the final benchmark artifacts.